In [ ]:
from google.colab import drive
drive.mount('/content/drive')
data = '/content/drive/MyDrive/universal_filter_ngspice/datasets/kaspix_ltc3417a_dataset.pt'

In [ ]:
import torch
cargar_data = torch.load(data)
# 1. Accedemos a la lista de entradas 'x'
lista_entradas = cargar_data['x']

# 2. Elegimos el circuito (por ejemplo, el primero)
idx = 0
circuito_info = lista_entradas[idx]

# 3. Extraemos los valores
knobs = circuito_info['knobs']
origen = circuito_info['netlist_origin']

print(f" Datos del circuito {idx} ({origen}):")
print("-" * 40)
print(f"Valores de los Knobs (Tensor): {knobs}")

# Mapeo típico para LTC3417A (según el orden en tus sensores de potencia)
# Generalmente el orden es: [Vin, Iload, R_feedback1, R_feedback2] o similar
nombres_params = ["Vin (V)", "Iload (A)", "R_fb1 (kΩ)", "R_fb2 (kΩ)"]

for i, valor in enumerate(knobs):
    nombre = nombres_params[i] if i < len(nombres_params) else f"Param_{i}"
    print(f"{nombre:<12} : {valor.item():.4f}")

In [ ]:
# 1. Obtenemos la señal de salida 'y' del primer circuito
señal = cargar_data['y'][0]

# 2. Contamos cuántas muestras (puntos) tiene
num_muestras = señal.shape[0]

# 3. Definimos la frecuencia de muestreo (FS)
# En tus proyectos anteriores hemos usado 48000 Hz
fs = 48000

# 4. Calculamos la duración total en segundos
duracion_segundos = num_muestras / fs

print(f"Número de muestras: {num_muestras}")
print(f"Frecuencia de muestreo: {fs} Hz")
print(f" Duración total: {duracion_segundos:.4f} segundos")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence

class KaspixPowerDataset(Dataset):
    def __init__(self, file_path):
        checkpoint = torch.load(file_path)
        self.x_data = checkpoint['x']
        self.y_data = checkpoint['y']

        all_y_points = torch.cat([y.flatten() for y in self.y_data[:100]])

        self.y_mean = torch.mean(all_y_points)
        self.y_std = torch.std(all_y_points)

        print(f" Estadísticas calculadas:")
        print(f"   Media (DC offset): {self.y_mean:.4f}V")
        print(f"   Desviación (Ripple scale): {self.y_std:.4f}V")

    def __len__(self):
        return len(self.x_data)

    def __getitem__(self, idx):
        x_seq = self.x_data[idx]['audio_in'].clone().detach().unsqueeze(0)
        knobs = self.x_data[idx]['knobs'].clone().detach()
        y_seq = self.y_data[idx].clone().detach().unsqueeze(0)

        y_seq_norm = (y_seq - self.y_mean) / self.y_std

        return x_seq, knobs, y_seq_norm

def kaspix_collate_fn(batch):
    x_seqs, knobs, y_seqs = zip(*batch)

    x_list = [x.t() for x in x_seqs]
    y_list = [y.t() for y in y_seqs]

    x_padded = pad_sequence(x_list, batch_first=True).transpose(1, 2)
    y_padded = pad_sequence(y_list, batch_first=True).transpose(1, 2)

    knobs_stacked = torch.stack(knobs)

    return x_padded, knobs_stacked, y_padded

# 1. Cargar Dataset Completo
full_ds = KaspixPowerDataset(data)
train_ds, val_ds, test_ds = random_split(full_ds, [0.7, 0.15, 0.15])

# LA CLAVE: Agregar collate_fn=kaspix_collate_fn
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, collate_fn=kaspix_collate_fn)
val_loader = DataLoader(val_ds, batch_size=8, collate_fn=kaspix_collate_fn)
test_loader = DataLoader(test_ds, batch_size=8, collate_fn=kaspix_collate_fn)

print(f" Partición 70/15/15 lista:")
print(f"   - Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

In [ ]:
import torch
import torch.nn as nn

# ============================================================
class FiLMLayer(nn.Module):
    """
    Inyecta los Knobs (L, C, R, Vin) en las características de la red.
    Ajusta la escala (gamma) y el desplazamiento (beta) de las activaciones.
    """
    def __init__(self, num_knobs, num_features):
        super().__init__()
        self.gamma = nn.Linear(num_knobs, num_features)
        self.beta = nn.Linear(num_knobs, num_features)

    def forward(self, x, knobs):
        # x: (Batch, Features, Length) o (Batch, Length, Features)
        # Determinar si la dimensión de características es la 1 (Conv) o la 2 (Recurrent)
        if x.shape[1] == self.gamma.out_features: # Caso Convolucional (B, C, L)
            g = self.gamma(knobs).unsqueeze(2)
            b = self.beta(knobs).unsqueeze(2)
        else: # Caso Recurrente (B, L, F)
            g = self.gamma(knobs).unsqueeze(1)
            b = self.beta(knobs).unsqueeze(1)
        return g * x + b

# ============================================================
class PowerTCN(nn.Module):
    def __init__(self, num_knobs=4):
        super().__init__()
        # Aumentamos kernel a 13 para capturar mejor la forma de la onda
        # Dilataciones exponenciales (1, 2, 4, 8) para ampliar el campo receptivo
        self.conv1 = nn.Conv1d(1, 64, kernel_size=13, padding=6)
        self.film1 = FiLMLayer(num_knobs, 64)

        self.conv2 = nn.Conv1d(64, 64, kernel_size=13, padding=12, dilation=2)
        self.film2 = FiLMLayer(num_knobs, 64)

        self.conv3 = nn.Conv1d(64, 64, kernel_size=13, padding=24, dilation=8)
        self.film3 = FiLMLayer(num_knobs, 64)

        self.conv4 = nn.Conv1d(64, 64, kernel_size=13, padding=48, dilation=32)
        self.film4 = FiLMLayer(num_knobs, 64)

        self.conv_out = nn.Conv1d(64, 1, kernel_size=1)

    def forward(self, x_seq, knobs):
        # x_seq: (B, 1, L)
        x = torch.relu(self.film1(self.conv1(x_seq), knobs))
        x = torch.relu(self.film2(self.conv2(x), knobs))
        x = torch.relu(self.film3(self.conv3(x), knobs))
        x = torch.relu(self.film4(self.conv4(x), knobs))

        return self.conv_out(x)

class PowerLSTM(nn.Module):
    def __init__(self, num_knobs=4):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=64, num_layers=2, batch_first=True)
        self.film = FiLMLayer(num_knobs, 64)
        self.fc = nn.Linear(64, 1)

    def forward(self, x_seq, knobs):
        # Transponer para LSTM: (B, 1, L) -> (B, L, 1)
        x = x_seq.transpose(1, 2)
        out, _ = self.lstm(x) # out: (B, L, 64)

        # Aplicamos FiLM sobre la salida de la LSTM
        out = self.film(out, knobs)

        out = self.fc(out) # out: (B, L, 1)
        return out.transpose(1, 2) # Volver a (B, 1, L)

class PowerRNN(nn.Module):
    def __init__(self, num_knobs=4):
        super().__init__()
        self.rnn = nn.RNN(input_size=1, hidden_size=64, num_layers=2, batch_first=True)
        self.film = FiLMLayer(num_knobs, 64)
        self.fc = nn.Linear(64, 1)

    def forward(self, x_seq, knobs):
        x = x_seq.transpose(1, 2)
        out, _ = self.rnn(x)

        out = self.film(out, knobs)

        out = self.fc(out)
        return out.transpose(1, 2)


In [ ]:
import numpy as np
import torch
from google.colab import drive
import os
import time

MODEL_SAVE_PATH = '/content/drive/MyDrive/universal_filter_ngspice/checkpoints/ltc/'
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

def train_kaspix_v2(model, train_loader, val_loader, epochs=60, patience=10):
    start_time = time.time()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    mse_criterion = torch.nn.MSELoss()
    mae_criterion = torch.nn.L1Loss()

    best_val_loss = float('inf')
    epochs_no_improve = 0

    print(f" Entrenando {model.__class__.__name__} con Early Stopping...")

    for epoch in range(epochs):
        # FASE DE ENTRENAMIENTO
        model.train()
        for x_seq, knobs, y_true in train_loader:
            x_seq, knobs, y_true = x_seq.to(device), knobs.to(device), y_true.to(device)

            optimizer.zero_grad()
            y_pred = model(x_seq, knobs)
            y_true = y_true[:, :, :y_pred.shape[2]]
            loss = mse_criterion(y_pred, y_true)
            loss.backward()
            optimizer.step()

        # FASE DE VALIDACIÓN
        model.eval()
        val_loss = 0
        val_mae = 0
        val_esr = 0
        with torch.no_grad():
            for x_seq, knobs, y_true in val_loader:
                x_seq, knobs, y_true = x_seq.to(device), knobs.to(device), y_true.to(device)
                y_pred = model(x_seq, knobs)
                y_true = y_true[:, :, :y_pred.shape[2]]

                mse = mse_criterion(y_pred, y_true)
                mae = mae_criterion(y_pred, y_true)

                error_sq = torch.sum((y_true - y_pred) ** 2)
                signal_sq = torch.sum(y_true ** 2)
                esr = error_sq / signal_sq

                val_loss += mse.item()
                val_mae += mae.item()
                val_esr += esr.item()

        # Calcular promedios
        avg_val_loss = val_loss / len(val_loader)
        avg_val_mae = val_mae / len(val_loader)
        avg_val_esr = val_esr / len(val_loader)

        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:03d} | MSE: {avg_val_loss:.6f} | MAE: {avg_val_mae:.6f}V | ESR: {avg_val_esr:.6f}")

        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step(avg_val_loss)

        # EARLY STOPPING Y GUARDADO
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
            # Guardar el mejor modelo
            save_file = os.path.join(MODEL_SAVE_PATH, f"best_{model.__class__.__name__}.pth")
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'loss_mse': best_val_loss,
                'mae': avg_val_mae,
                'esr': avg_val_esr
            }, save_file)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f" Early Stopping en epoch {epoch+1}. El modelo no mejoró por {patience} épocas.")
                break
    total_time = (time.time() - start_time) / 60
    print(f"\n Entrenamiento completado en {total_time} s.")

In [ ]:
import time
import os

def train_kaspix_v3(model, train_loader, val_loader, epochs=100, patience=15):
    start_time = time.time()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    mse_criterion = torch.nn.MSELoss()
    mae_criterion = torch.nn.L1Loss()

    best_val_loss = float('inf')
    epochs_no_improve = 0

    print(f" Entrenando {model.__class__.__name__} con Normalización y Scheduler...")

    for epoch in range(epochs):
        # FASE DE ENTRENAMIENTO
        model.train()
        for x_seq, knobs, y_true in train_loader:
            x_seq, knobs, y_true = x_seq.to(device), knobs.to(device), y_true.to(device)

            optimizer.zero_grad()
            y_pred = model(x_seq, knobs)
            # Asegurar que las longitudes coincidan por el padding del TCN
            y_true = y_true[:, :, :y_pred.shape[2]]
            loss = mse_criterion(y_pred, y_true)
            loss.backward()
            optimizer.step()

        # FASE DE VALIDACIÓN
        model.eval()
        val_loss = 0
        val_mae = 0
        val_esr = 0
        with torch.no_grad():
            for x_seq, knobs, y_true in val_loader:
                x_seq, knobs, y_true = x_seq.to(device), knobs.to(device), y_true.to(device)
                y_pred = model(x_seq, knobs)
                y_true = y_true[:, :, :y_pred.shape[2]]

                mse = mse_criterion(y_pred, y_true)
                mae = mae_criterion(y_pred, y_true)

                error_sq = torch.sum((y_true - y_pred) ** 2)
                signal_sq = torch.sum(y_true ** 2)
                esr = error_sq / (signal_sq + 1e-7) # Evitar división por cero

                val_loss += mse.item()
                val_mae += mae.item()
                val_esr += esr.item()

        # Calcular promedios de la época
        avg_val_loss = val_loss / len(val_loader)
        avg_val_mae = val_mae / len(val_loader)
        avg_val_esr = val_esr / len(val_loader)

        # ACTUALIZAR SCHEDULER: Ahora avg_val_loss ya está definida
        scheduler.step(avg_val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:03d} | MSE: {avg_val_loss:.6f} | ESR: {avg_val_esr:.6f} | LR: {current_lr:.6f}")

        # EARLY STOPPING Y GUARDADO
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
            save_file = os.path.join(MODEL_SAVE_PATH, f"best_{model.__class__.__name__}.pth")
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'loss_mse': best_val_loss,
                'mae': avg_val_mae,
                'esr': avg_val_esr
            }, save_file)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f" Early Stopping en epoch {epoch+1}. Sin mejora por {patience} épocas.")
                break

    total_time = (time.time() - start_time) / 60
    print(f"\n Entrenamiento completado en {total_time:.2f} minutos.")


In [ ]:
model_tcn  = PowerTCN(num_knobs=4)
train_kaspix_v2(model_tcn, train_loader, val_loader, epochs=100, patience=15)

In [ ]:
model_lstm = PowerLSTM(num_knobs=4)
train_kaspix_v2(model_lstm, train_loader, val_loader, epochs=80, patience=15)

In [ ]:
model_rnn  = PowerRNN(num_knobs=4)
train_kaspix_v2(model_rnn, train_loader, val_loader, epochs=80, patience=15)

Gráficos

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

# 1. Configuración de dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def visualize_kaspix_results(model_class, dataset, device, model_path, idx=None):
    # Instanciar el modelo y cargar pesos
    model = model_class(num_knobs=4).to(device)
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    # Si idx es None, elige uno al azar. Si no, usa el que pases.
    if idx is None:
        idx = np.random.randint(len(dataset))

    # 2. Obtención de datos
    x_seq_raw, knobs_raw, y_true_raw = dataset[idx]

    x_seq = x_seq_raw.unsqueeze(0).to(device).float()
    knobs = knobs_raw.unsqueeze(0).to(device).float()
    target = y_true_raw.squeeze().cpu().numpy()

    # 3. Inferencia
    with torch.no_grad():
        # El modelo recibe (x_seq, knobs)
        y_pred = model(x_seq, knobs)
        prediction = y_pred.cpu().squeeze().numpy()

    # Ajuste de longitud por si hay diferencias de padding
    min_len = min(len(target), len(prediction))
    target = target[:min_len]
    prediction = prediction[:min_len]

    # 4. Graficación sin zoom
    plt.figure(figsize=(15, 10))

    # Subplot 1: Comparación
    plt.subplot(2, 1, 1)
    plt.plot(target, label='Target (NGSPICE)', color='#1f77b4', lw=2, alpha=0.8)
    plt.plot(prediction, label=f'{model_class.__name__} Prediction', color='#ff7f0e', linestyle='--', lw=1.5)

    nombres = ["Vin", "Iload", "Rfb1", "Rfb2"]
    knobs_info = ", ".join([f"{n}:{v:.2f}" for n, v in zip(nombres, knobs_raw)])

    plt.title(f"LTC3417A - Sample Index: {idx}\nParams: [{knobs_info}]", fontsize=14)
    plt.ylabel("Amplitude (V/A)")
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)

    # Subplot 2: Error
    plt.subplot(2, 1, 2)
    error = target - prediction
    plt.fill_between(range(len(error)), error, color='red', alpha=0.2)
    plt.plot(error, color='red', lw=0.8)

    mae = np.abs(error).mean()
    plt.title(f"Residual Error (MAE: {mae:.6f}V)", fontsize=12)
    plt.xlabel("Time Samples")
    plt.ylabel("Delta Error")
    plt.axhline(0, color='black', lw=1, alpha=0.5)
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
MODEL_PATH_tcn = '/content/drive/MyDrive/universal_filter_ngspice/checkpoints/ltc/best_PowerTCN.pth'
visualize_kaspix_results(PowerTCN, test_ds, device, MODEL_PATH_tcn, idx=7)

In [ ]:
MODEL_PATH_lstm = '/content/drive/MyDrive/universal_filter_ngspice/checkpoints/ltc/best_PowerLSTM.pth'
visualize_kaspix_results(PowerLSTM, test_ds, device, MODEL_PATH_lstm, idx=7)

In [ ]:
MODEL_PATH_rnn = '/content/drive/MyDrive/universal_filter_ngspice/checkpoints/ltc/best_PowerRNN.pth'
visualize_kaspix_results(PowerRNN, test_ds, device, MODEL_PATH_rnn, idx=7)